# Packages

In [20]:
import sys
print(sys.executable)
print(sys.version)


/usr/bin/python3
3.10.12 (main, Aug 15 2025, 14:32:43) [GCC 11.4.0]


# Imports

In [36]:
from pypdf import PdfReader
import numpy as np
print(np.__version__)
import networkx as nx
#import matplotlib.pyplot as plt
import pandas as pd
import graphviz
from unidecode import unidecode
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from nltk.stem import WordNetLemmatizer
from nltk.util import ngrams
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
from re import sub as regexSubstitute
from cleantext import clean
#import cleantext
import io
import re
import string
import tqdm
from gensim.models import Word2Vec
import tensorflow as tf
#from tensorflow.keras import layers


1.26.4


In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Octen/Octen-Embedding-4B")

# Encode sentences
sentences = [
    "This is an example sentence",
    "Each sentence is converted to a vector"
]

embeddings = model.encode(sentences)
print(embeddings.shape)
# Output: (2, 2560)

# Compute similarity
from sentence_transformers.util import cos_sim
similarity = cos_sim(embeddings[0], embeddings[1])
print(f"Similarity: {similarity.item():.4f}")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

(2, 2560)
Similarity: 0.4394


# Function Definitions

In [23]:
#Converts POS tags to accepted format
def tagConversion(tag):
    tagsetMapping = {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV
    }

    start = tag[0].upper()
    tag = tagsetMapping.get(start)
    if tag == None: tag = wordnet.NOUN

    return tag

#Creates structures with frequency to get the graph
def freqArray(pairs):
    uniquePairs, counts = np.unique(pairs, axis = 0, return_counts = True)
    counts = counts.reshape(-1, 1)
    pairFreqs = np.concatenate((uniquePairs, counts), axis = 1)
    pairFreqs = np.array(sorted(pairFreqs, key=lambda x: -x[2]))

    return pairFreqs

#Calculates the cosine similarity between 2 vectors
def cosSim(a, b):
    res = np.dot(a, b)/(np.linalg.norm(a) * np.linalg.norm(b))
    return res

def conMatrix(pairfreqs):
    dim = len(pairfreqs)
    matrix = np.zeros((dim, dim))
    for i in range(dim):
        matrix[pairfreqs[i][0], pairfreqs[i][1]] = pairfreqs[i][2]

    return matrix


# NLTK Downloads

In [24]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('universal_tagset')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/miguelfsoto/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/miguelfsoto/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/miguelfsoto/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/miguelfsoto/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/miguelfsoto/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/miguelfsoto/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/

True

# Get the raw text from the PDF
The following is for an article obtained from JSTOR \
Processing of text from a different source would require different treatment

In [25]:
reader = PdfReader("jstor-article.pdf")
corpus = ""
for page in reader.pages[1:]:
    raw = page.extract_text()
    raw = raw.split()
    cutoff = 0
    for i in range(len(raw)):
        if raw[i] == "This" and raw[i + 1] == "content":
            cutoff = i
            break
    raw = raw[:cutoff]
    raw = " ".join(raw)
    corpus += raw
corpus = corpus[0:corpus.index("References")]
#print(corpus)

This section used the cleantext package to remove certain elements from the text \
Following an update, this no longer works

In [26]:
#corpus = clean(corpus,
               #fix_unicode=True,               # fix various unicode errors
               #to_ascii=True,                  # transliterate to closest ASCII representation
               #lower=True,                     # lowercase text
               #no_line_breaks=True,           # fully strip line breaks as opposed to only normalizing them
               #no_urls=True,                  # replace all URLs with a special token
               #no_emails=True,                # replace all email addresses with a special token
               #no_phone_numbers=True,         # replace all phone numbers with a special token
               #no_numbers=True,               # replace all numbers with a special token
               #no_digits=True,                # replace all digits with a special token
               #no_currency_symbols=True,      # replace all currency symbols with a special token
               #no_punct=True,                 # remove punctuations
               #)
#print(corpus)

# Handling of short words and line breaks
The text contains many instances of words being separated across two lines when they would not fit into one \
The following section handles this, as well as words separated by a dash and links

In [27]:
corpus = corpus.split()
corpus = [s for s in corpus if s[0] != '<' and s[-1] != '0' and len(s) > 2]
for i in range(len(corpus)):
    word = corpus[i]
    if word == '':
        continue
    if word[:4] == "http":
        corpus[i] = ""
    if word[-1] == "-":
        word = word[:-1]
        word += corpus[i + 1]
        corpus[i] = word
        corpus[i + 1] = ''
corpus = ' '.join(corpus)
corpus = corpus.replace("-", " ")
print(corpus)

They Endorse It, Can't Trust It: How Outgroup Leader Endorsements Undercut Public Support for Civil War Peace Settlements Nicholas Haas New York University Prabin Khadka New York University Abstract: Civil wars are greater source violence than any other type conflict, yet little known about one the key determinants civil war peace settlement success: civilian support evaluate how core component nearly all peace settlements, leader endorsements, affects public support. predict that individuals conflict settings will view settlements endorsed outgroup leaders less trustworthy and that they will become less supportive. conduct endorsement experiment with nearly respondents South Sudan 2016, taking advantage brief cessation devastating civil war. Public support for tentative settlement drops precipitously when endorsed outgroup leader but does not increase when endorsed ingroup leader. find suggestive evidence that effects are strongest for individuals with the greatest reason fear outgrou

# Tokenization

In [28]:
corpus = unidecode(corpus.lower())
corpus = regexSubstitute(r'[^a-z\s]', '', corpus)
corpus = regexSubstitute(r'\s+', ' ', corpus).split()

stopWords = set(stopwords.words('english'))
stopWords.update(['us','whose'])
tokenCorpus = word_tokenize(str.join(" ",[x for x in corpus if x not in stopWords and len(x) >= 2]))
#print(tokenCorpus)

# Tagging and Lemmatization

In [29]:
patternTags = nltk.pos_tag(tokenCorpus)

lemmatizer = WordNetLemmatizer()
lemmas = []
vocab = set()
for token, tag in patternTags:
    lemma = lemmatizer.lemmatize(token, pos = tagConversion(tag))
    lemmas.append(lemma)
    vocab.add(lemma)
print(lemmas)

['endorse', 'cant', 'trust', 'outgroup', 'leader', 'endorsement', 'undercut', 'public', 'support', 'civil', 'war', 'peace', 'settlement', 'nicholas', 'haas', 'new', 'york', 'university', 'prabin', 'khadka', 'new', 'york', 'university', 'abstract', 'civil', 'war', 'great', 'source', 'violence', 'type', 'conflict', 'yet', 'little', 'known', 'one', 'key', 'determinant', 'civil', 'war', 'peace', 'settlement', 'success', 'civilian', 'support', 'evaluate', 'core', 'component', 'nearly', 'peace', 'settlement', 'leader', 'endorsement', 'affect', 'public', 'support', 'predict', 'individual', 'conflict', 'setting', 'view', 'settlement', 'endorse', 'outgroup', 'leader', 'less', 'trustworthy', 'become', 'less', 'supportive', 'conduct', 'endorsement', 'experiment', 'nearly', 'respondent', 'south', 'sudan', 'take', 'advantage', 'brief', 'cessation', 'devastate', 'civil', 'war', 'public', 'support', 'tentative', 'settlement', 'drop', 'precipitously', 'endorse', 'outgroup', 'leader', 'increase', 'endo

# Frequency data and word vectors

In [31]:
tokenCorpus = lemmas

frequency = FreqDist(tokenCorpus)

wordIndex = {}
#indexWord = {0: '<pad>'}
indexWord = {}
index = 0
checked = []
for word in tokenCorpus:
    if word not in checked:
        wordIndex[word] = index
        indexWord[index] = word
        index += 1
        checked.append(word)
#print(wordIndex)
# print(len(wordIndex))
seq = [wordIndex[word] for word in tokenCorpus]
#print("seq", seq)
# print(tokenCorpus)
ngr = ngrams(tokenCorpus, 2)
ngrList = [list(x) for x in list(ngr)]

w2v = Word2Vec(ngrList, min_count=1)
print("going in")
#for word in tokenCorpus:
#    print(word)
#    emb = model.encode(word)
#    print(emb)
#emb = model.encode(tokenCorpus)
#print(emb.shape)
#w2v.build_vocab(ngr)
print(len(w2v.wv))
totalLen = len(wordIndex)
print(totalLen)

going in
1499
1499


# PMI Calculation

In [52]:
def pmi(w1, w2):
    bigram_measures = BigramAssocMeasures()
    finder = BigramCollocationFinder.from_words(tokenCorpus)
    for i in finder.score_ngrams(bigram_measures.pmi):
        #print(i)
        if i[0] == (w1, w2):
            #print("match")
            return i[1] 

print(pmi("take", "land"))

9.669897401880348


# Check Cosine Similarity and PMI

In [58]:
edges = []
for i in range(len(indexWord)):
    for j in range(i, len(indexWord)):
        word1 = indexWord[i]
        word2 = indexWord[j]
        pmiWords = pmi(word1, word2)
        wv1 = w2v.wv[wordIndex[word1]]
        wv2 = w2v.wv[wordIndex[word2]]
        cosWords = cosSim(wv1, wv2)
        if pmiWords != None and pmiWords > 6 and cosWords > 0:
            print(word1, word2, pmiWords, cosWords)
            edges.append((word1, word2))

endorse cant 8.669897401880348 0.14635324
cant trust 8.229324810494367 0.08689075
trust realize 8.229324810494367 0.16506197
trust build 8.229324810494367 0.096951805
outgroup verification 6.017820705300655 0.26771036
leader intention 6.06786138780025 0.051277313
leader currently 6.06786138780025 0.040178273
leader train 6.06786138780025 0.0652853
leader describe 6.06786138780025 0.14107788
leader conveys 6.06786138780025 0.09363147
undercut public 8.30732732249564 0.21443692
public commercial 8.30732732249564 0.07143857
civil war 7.783765366438628 0.1330239
war assume 7.953690367880939 0.12295172
settlement success 6.482898887079094 0.3475172
settlement specifically 6.804826981966457 0.019577531
settlement signature 6.804826981966457 0.10476021
settlement ppi 6.804826981966457 0.10194085
nicholas haas 9.155324229050592 0.09832581
haas prabin 8.833396134163227 0.084393576
haas phd 8.155324229050589 0.022799844
new york 8.669897401880348 0.14552931
new class 8.669897401880348 0.09896269

# Creation of contexts
The context of a word is given by the words before and after it in a given window \
This window is set to 2 in this example \
The context is then defined as the sum of the word vectors of the words in the window (not including the target)

These context vectors are then compared against each other using the cosine similarity criteria \
Those above a certain similarity threshold (0.4 in this example) are then marked as connected for graph creation

In [32]:
wordContexts = []
for i in range(len(tokenCorpus)):
    #print(tokenCorpus[i])
    #print(cosSim(w2v.wv[i], w2v.wv[i + 1]))
    context = []
    if i > 2 and i < len(tokenCorpus) - 2:
        context = [tokenCorpus[i - 2], tokenCorpus[i - 1], tokenCorpus[i + 1], tokenCorpus[i + 2]]
    elif i < 2:
        context = [tokenCorpus[i + 1], tokenCorpus[i + 2]]
    else:
        context = [tokenCorpus[i - 1], tokenCorpus[i - 2]]
    # print(context)
    agg = w2v.wv[wordIndex[context[0]]].copy()
    #print(agg)
    for word in context[1:]:
        for j in range(len(agg)):
            agg[j] += w2v.wv[wordIndex[word]][j]
    # print(context)
    wordContexts.append((tokenCorpus[i], agg))
print("ctx done")
grid = []
for i in range(len(wordContexts)):
    for j in range(i, len(wordContexts)):
        word1 = wordContexts[i][0]
        word2 = wordContexts[j][0]
        if word1 != word2:
            wordSim = cosSim(wordContexts[i][1], wordContexts[j][1])
            if wordSim >= 0.4:
                grid.append([word1, word2, wordSim])
print(len(grid))

ctx done
315587


# Dataframe and Graph Creation
A dataframe and graph are then created with the words that were connected, as well as the similarity index \
The graph data is exported to a graphviz file to be loaded into Gephi for visualization

In [33]:
dataframe = pd.DataFrame(grid)
dataframe.columns = ['A', 'B', 'Weight']
print(dataframe)

dot = graphviz.Graph('Word Graph', engine='neato')
graph = nx.Graph()
print("graph done")

# Create the data frame
for _, row in dataframe.iterrows():
    a = row['A']
    b = row['B']
    w = row['Weight']
    #print(a, b, w)
    graph.add_edge(a, b, weight = w)

    dot.node(str(row['A']), row['A'])
    dot.node(str(row['B']), row['B'])
    dot.edge(str(row['A']), str(row['B']), weight= str(w))
print("dataframe done")

dot.render(filename="./wordgraph.gv")

               A         B    Weight
0        endorse      cant  0.619160
1        endorse     trust  0.592222
2        endorse  outgroup  0.773520
3        endorse    leader  0.534208
4        endorse   success  0.409906
...          ...       ...       ...
315582  strategy    public  0.444905
315583  strategy   support  0.790690
315584  increase    public  0.433122
315585  increase   support  0.495316
315586    public   support  0.605559

[315587 rows x 3 columns]
graph done
dataframe done


'wordgraph.gv.pdf'

# Graph Metrics
Additional graph metrics are calculated and exported into a .csv file \
For unknown reasons, betweenness and closeness seem to not work (or just take an unmanageable amount of time), while clustering takes too long for the calculation to be done in a reasonable amount of time

In [34]:
positions = nx.spring_layout(graph)
print("layout done")
weights = nx.get_edge_attributes(graph, 'weight')
print("attr done")

for node in graph.nodes:
    shortest_path = nx.shortest_path(graph, source=node, weight='weight')
    #print("sp done")
    degree_centrality = nx.degree_centrality(graph)
    #print("cen done")
    #betweenness_centrality = nx.betweenness_centrality(graph, weight='weight')
    #print("btw done")
    #closeness_centrality = nx.closeness_centrality(graph, distance='weight')
    #print("cls done")
    #clustering_coefficient = nx.clustering(graph, weight='weight')
    #print("cluster done")
print("loop done")

layout done
attr done
loop done


In [35]:
def class_quartiles(df,name):
    name1 = 'Class_' + name + 'low'
    name2 = 'Class_' + name + 'medium'
    name3 = 'Class_' + name + 'high'
    quartiles = np.percentile(df[name], [25, 50, 75])
    df[name1] = np.where(df[name] >= quartiles[2], 1, 0)
    df[name2] = np.where((df[name] > quartiles[0]) & (df[name] < quartiles[2]), 1, 0)
    df[name3] = np.where((df[name] <= quartiles[0]) | (df[name] >= quartiles[2]), 1, 0)
    return df

df_metrics = pd.DataFrame(indexWord.items(), columns=['Node', 'word'])
df_metrics['frequency_word'] = df_metrics['word'].map(frequency)
df_metrics['shortest_path'] = df_metrics['Node'].map(shortest_path)
df_metrics['degree_centrality'] = df_metrics['Node'].map(degree_centrality)
#df_metrics['betweenness_centrality'] = df_metrics['Node'].map(betweenness_centrality)
#df_metrics['closeness_centrality'] = df_metrics['Node'].map(closeness_centrality)
#df_metrics['clustering_coefficient'] = df_metrics['Node'].map(clustering_coefficient)
df_metrics['eigenvector_centrality'] = df_metrics['Node'].map(nx.eigenvector_centrality(graph, weight='weight'))
df_metrics = df_metrics.sort_values("eigenvector_centrality", ascending=False)


df_metrics=class_quartiles(df_metrics,'degree_centrality')
#df_metrics=class_quartiles(df_metrics,'betweenness_centrality')
#df_metrics=class_quartiles(df_metrics,'closeness_centrality')
#df_metrics=class_quartiles(df_metrics,'clustering_coefficient')
df_metrics=class_quartiles(df_metrics,'eigenvector_centrality')

## save the metrics in a csv file
df_metrics.to_csv('metrics.csv', index=False)

## create the dataframe only with the word and all columns beging Class

df_metrics_class=df_metrics.filter(regex='Class', axis=1)

# df_metric aggregate the metrics of each word in THE FIRST COLUMN
df_metrics_class['word']=df_metrics['word']

columnas = df_metrics_class.columns.tolist()
columnas = [columnas[-1]] + columnas[:-1]
df_metrics_class = df_metrics_class[columnas]

# save the dataframe in a csv file
df_metrics_class.to_csv('metrics_class.csv', index=False)
print("metrics done")

metrics done


/tmp/ipykernel_74295/2720289642.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_metrics_class['word']=df_metrics['word']
